# HealthBot: AI-Powered Patient Education (LangGraph Prototype)

This notebook runs an interactive HealthBot workflow:
- asks for a health topic
- searches reputable sources via Tavily
- summarizes results in patient-friendly language with citations
- delivers a single-question quiz and feedback with citations


## Imports
Minimal imports for the workflow. Install dependencies with `pip install -r requirements.txt`.

In [ ]:
import json
import os
from typing import Any, Dict, List, Optional, TypedDict
from urllib.parse import urlparse

from dotenv import load_dotenv
from langchain_cohere import ChatCohere
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import END, StateGraph


## Environment and tools
Keys, model, and search setup.

In [ ]:
load_dotenv()

COHERE_API_KEY = os.getenv("COHERE_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

if not COHERE_API_KEY:
    raise RuntimeError("COHERE_API_KEY is not set in the environment.")
if not TAVILY_API_KEY:
    raise RuntimeError("TAVILY_API_KEY is not set in the environment.")

llm = ChatCohere(
    model="command-a-03-2025",
    temperature=0.2,
    cohere_api_key=COHERE_API_KEY,
)
search_tool = TavilySearchResults(max_results=5)

REPUTABLE_DOMAINS = [
    "cdc.gov",
    "nih.gov",
    "who.int",
    "mayoclinic.org",
    "medlineplus.gov",
    "nhs.uk",
    "clevelandclinic.org",
    "hopkinsmedicine.org",
]


## State schema and helpers
State shape and small utility helpers.

In [ ]:
class HealthState(TypedDict, total=False):
    topic: str
    search_results: List[Dict[str, Any]]
    summary: str
    quiz_question: str
    quiz_answer: str
    patient_answer: str
    grade: str
    feedback: str
    continue_session: bool


def _clear_session_state() -> Dict[str, Any]:
    return {
        "search_results": [],
        "summary": "",
        "quiz_question": "",
        "quiz_answer": "",
        "patient_answer": "",
        "grade": "",
        "feedback": "",
    }


def _print_header(title: str) -> None:
    line = "=" * 72
    print(f"\n{line}\n{title}\n{line}")


def _print_subheader(title: str) -> None:
    line = "-" * 72
    print(f"\n{title}\n{line}")


def _build_query(topic: str) -> str:
    site_filters = " OR ".join(f"site:{domain}" for domain in REPUTABLE_DOMAINS)
    return f"{topic} ({site_filters})"


def _filter_reputable(results: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    filtered: List[Dict[str, Any]] = []
    for result in results:
        url = result.get("url", "")
        netloc = urlparse(url).netloc.lower()
        if any(domain in netloc for domain in REPUTABLE_DOMAINS):
            filtered.append(result)
    return filtered


def _format_sources(results: List[Dict[str, Any]]) -> str:
    lines = []
    for idx, result in enumerate(results, start=1):
        title = result.get("title", "Untitled")
        url = result.get("url", "")
        content = result.get("content", "")
        lines.append(f"[{idx}] {title}\nURL: {url}\nSnippet: {content}\n")
    return "\n".join(lines)


def _format_citation_links(results: List[Dict[str, Any]]) -> str:
    lines = []
    for idx, result in enumerate(results, start=1):
        url = result.get("url", "")
        if url:
            lines.append(f"[{idx}] {url}")
    return "\n".join(lines)


def _safe_json_loads(text: str) -> Dict[str, Any]:
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        start = text.find("{")
        end = text.rfind("}")
        if start != -1 and end != -1 and end > start:
            try:
                return json.loads(text[start : end + 1])
            except json.JSONDecodeError:
                return {}
        return {}


## Topic intake and search
Prompt the patient and call Tavily.

In [ ]:
def ask_topic(_: HealthState) -> Dict[str, Any]:
    while True:
        topic = input("What health topic or condition would you like to learn about? ").strip()
        if topic:
            break
        print("Please enter a topic.")
    return {"topic": topic, **_clear_session_state()}


def search_tavily(state: HealthState) -> Dict[str, Any]:
    topic = state.get("topic", "")
    query = _build_query(topic)
    results = search_tool.invoke({"query": query})
    filtered = _filter_reputable(results)
    if not filtered:
        filtered = results
    return {"search_results": filtered}


## Summarization and readiness gate
Summarize sources and wait for readiness.

In [ ]:
def summarize(state: HealthState) -> Dict[str, Any]:
    topic = state.get("topic", "")
    results = state.get("search_results", [])
    sources_text = _format_sources(results)
    prompt = (
        "You are a patient educator. Summarize the information in plain language, "
        "avoiding medical jargon when possible. Use 3-6 short paragraphs or bullets. "
        "Include citations like [1], [2] that map to the source list. "
        "If the sources are limited, be transparent about uncertainty."
    )
    response = llm.invoke(
        [
            ("system", prompt),
            (
                "user",
                f"Topic: {topic}\n\nSources:\n{sources_text}\n\nSummary:",
            ),
        ]
    )
    summary = response.content.strip()
    return {"summary": summary}


def present_summary(state: HealthState) -> Dict[str, Any]:
    topic = state.get("topic", "")
    summary = state.get("summary", "")
    _print_header("HealthBot Summary")
    if topic:
        _print_subheader("Topic")
        print(topic)
    _print_subheader("Summary")
    print(summary)
    return {"summary": summary}


def wait_ready(state: HealthState) -> Dict[str, Any]:
    while True:
        ready = input("Type 'ready' when you're ready for a quick check: ").strip().lower()
        if ready in {"ready", "r", "yes", "y"}:
            break
        print("No problem. Take your time, then type 'ready'.")
    return {"summary": state.get("summary", "")}


## Quiz and answer capture
Create a quiz and collect the answer.

In [ ]:
def create_quiz(state: HealthState) -> Dict[str, Any]:
    summary = state.get("summary", "")
    prompt = (
        "Create exactly one comprehension question based on the summary. "
        "Provide a short expected answer (1-2 sentences). "
        "Return JSON with keys 'question' and 'answer'."
    )
    response = llm.invoke(
        [
            ("system", prompt),
            ("user", summary),
        ]
    )
    data = _safe_json_loads(response.content)
    question = data.get("question") or "What is one key takeaway from the summary?"
    answer = data.get("answer") or "A short, accurate answer based on the summary."
    return {"quiz_question": question, "quiz_answer": answer}


def ask_quiz(state: HealthState) -> Dict[str, Any]:
    question = state.get("quiz_question", "")
    _print_subheader("Quick Check")
    print(question)
    return {"quiz_question": question}


def get_answer(_: HealthState) -> Dict[str, Any]:
    answer = input("Your answer: ").strip()
    return {"patient_answer": answer}


## Grading and feedback
Assess the answer with citations.

In [ ]:
def grade_answer(state: HealthState) -> Dict[str, Any]:
    summary = state.get("summary", "")
    expected = state.get("quiz_answer", "")
    patient = state.get("patient_answer", "")
    prompt = (
        "You are a helpful tutor. Grade the patient's answer against the expected "
        "answer and the summary. Return JSON with keys 'grade' (Correct, Partially correct, "
        "Incorrect) and 'explanation' (1-3 sentences with citations like [1])."
    )
    response = llm.invoke(
        [
            ("system", prompt),
            (
                "user",
                f"Summary:\n{summary}\n\nExpected Answer:\n{expected}\n\n"
                f"Patient Answer:\n{patient}\n\nGrade:",
            ),
        ]
    )
    data = _safe_json_loads(response.content)
    grade = data.get("grade") or "Unclear"
    explanation = data.get("explanation") or response.content.strip()
    return {"grade": grade, "feedback": explanation}


def present_grade(state: HealthState) -> Dict[str, Any]:
    grade = state.get("grade", "")
    feedback = state.get("feedback", "")
    _print_header("Quiz Feedback")
    if grade:
        print(f"Grade: {grade}")
    if feedback:
        _print_subheader("Feedback")
        print(feedback)
    return {"grade": grade}


## Restart and reset
Offer another topic and clear state.

In [ ]:
def ask_restart(state: HealthState) -> Dict[str, Any]:
    while True:
        choice = input("Would you like to learn about another topic? (yes/no): ").strip().lower()
        if choice in {"yes", "y"}:
            return {"continue_session": True}
        if choice in {"no", "n"}:
            citations = _format_citation_links(state.get("search_results", []))
            _print_header("Sources")
            if citations:
                print(citations)
            else:
                print("No sources available.")
            print("\nThank you for using HealthBot.")
            return {"continue_session": False}
        print("Please answer 'yes' or 'no'.")


def reset_state(_: HealthState) -> Dict[str, Any]:
    cleared = _clear_session_state()
    cleared.update(
        {
            "topic": "",
            "continue_session": False,
        }
    )
    return cleared


## Routing helper
Decide whether to continue or end.

In [ ]:
def _should_continue(state: HealthState) -> str:
    return "continue" if state.get("continue_session") else "end"


## Graph wiring
Connect nodes and compile the app.

In [ ]:
graph = StateGraph(HealthState)

graph.add_node("ask_topic", ask_topic)
graph.add_node("search_tavily", search_tavily)
graph.add_node("summarize", summarize)
graph.add_node("present_summary", present_summary)
graph.add_node("wait_ready", wait_ready)
graph.add_node("create_quiz", create_quiz)
graph.add_node("ask_quiz", ask_quiz)
graph.add_node("get_answer", get_answer)
graph.add_node("grade_answer", grade_answer)
graph.add_node("present_grade", present_grade)
graph.add_node("ask_restart", ask_restart)
graph.add_node("reset_state", reset_state)

graph.set_entry_point("ask_topic")
graph.add_edge("ask_topic", "search_tavily")
graph.add_edge("search_tavily", "summarize")
graph.add_edge("summarize", "present_summary")
graph.add_edge("present_summary", "wait_ready")
graph.add_edge("wait_ready", "create_quiz")
graph.add_edge("create_quiz", "ask_quiz")
graph.add_edge("ask_quiz", "get_answer")
graph.add_edge("get_answer", "grade_answer")
graph.add_edge("grade_answer", "present_grade")
graph.add_edge("present_grade", "ask_restart")

graph.add_conditional_edges(
    "ask_restart",
    _should_continue,
    {
        "continue": "reset_state",
        "end": END,
    },
)

graph.add_edge("reset_state", "ask_topic")

app = graph.compile()


## Run the app
Invoke the workflow.

In [ ]:
app.invoke({"topic": " "})